# Topic 03 — Decision Trees vs k-NN

**Question.** How do model assumptions change validation quality? We compare an interpretable tree with a distance-based k-NN pipeline on the same split.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
X.shape, y.value_counts(normalize=True).round(3)


## Tree depth
A deeper tree can represent more complicated rules, but training flexibility increases variance. I will inspect test performance across several depths instead of assuming that deeper is better.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    rows.append({'model': f'tree depth={depth}', 'train_acc': model.score(X_train, y_train), 'test_acc': model.score(X_test, y_test)})
pd.DataFrame(rows).round(3)


## k-NN requires a meaningful distance
Because features use different physical scales, raw Euclidean distance would let large-scale columns dominate. The scaler belongs inside the pipeline so validation never sees preprocessing fitted on test data.

In [ ]:
results = []
for k in [3, 5, 9, 15]:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({'k': k, 'accuracy': accuracy_score(y_test, pred), 'balanced_accuracy': balanced_accuracy_score(y_test, pred)})
pd.DataFrame(results).round(3)


## Takeaway
The useful comparison is not ‘which algorithm wins forever?’ but ‘which inductive bias fits this dataset under a fair validation protocol?’. The untouched test set estimates generalization only after choices have been made.